In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 1. Cargar los datos
try:
    X = np.load('X.npy')
    Y = np.load('Y.npy')
    print("¡Datos cargados exitosamente!")
except FileNotFoundError:
    print("Error: No se encontraron los archivos. Revisa la ruta.")

In [ ]:
# 2. Análisis de las dimensiones
print(f"Total de imágenes: {X.shape[0]}")
print(f"Resolución de cada imagen: {X.shape[1]}x{X.shape[2]} píxeles")
print(f"Formato de la matriz de imágenes (X): {X.shape}")
print(f"Formato de la matriz de etiquetas (Y): {Y.shape}")

In [ ]:
# 3. Gráfico de distribución de clases
# Convertimos las etiquetas One-Hot de vuelta a números enteros (0-9) para contarlas
labels = np.argmax(Y, axis=1)
unique_classes, counts = np.unique(labels, return_counts=True)

plt.figure(figsize=(10, 5))
plt.bar(unique_classes, counts, color='teal', edgecolor='black')
plt.title("Distribución de Imágenes por Dígito (Lenguaje de Señas)", fontsize=14)
plt.xlabel("Dígito Representado", fontsize=12)
plt.ylabel("Cantidad de Imágenes", fontsize=12)
plt.xticks(unique_classes)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

In [ ]:
# 4. Muestra visual del dataset (Grilla 2x5)
plt.figure(figsize=(12, 6))

for i in range(10):
    # Encontramos el índice de la primera imagen que corresponde al dígito 'i'
    index = np.where(labels == i)[0][0]
    
    plt.subplot(2, 5, i + 1)
    # Las imágenes en este dataset suelen estar en escala de grises
    plt.imshow(X[index], cmap='gray')
    plt.title(f"Dígito: {i}", fontsize=12, fontweight='bold')
    plt.axis('off') # Ocultamos los ejes para que se vea más limpio

plt.tight_layout()
plt.suptitle("Muestra Aleatoria del Dataset Sign Language Digits", fontsize=16, y=1.05)
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Añadir la dimensión del canal (Grayscale = 1 canal)
# Las CNN en Keras esperan tensores con forma (lote, alto, ancho, canales)
X_cnn = X.reshape(-1, X.shape[1], X.shape[2], 1)

# 2. Normalización de píxeles
# Llevamos los valores de intensidad al rango de 0 a 1 para estabilizar los gradientes
X_cnn = X_cnn.astype('float32') / 255.0

# 3. División estratificada (80% Entrenamiento, 10% Validación, 10% Prueba)
X_train, X_temp, y_train, y_temp = train_test_split(X_cnn, Y, test_size=0.2, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print(f"Muestras de Entrenamiento: {X_train.shape[0]}")
print(f"Muestras de Validación: {X_val.shape[0]}")
print(f"Muestras de Prueba: {X_test.shape[0]}")
print(f"Nueva forma del tensor de entrada: {X_train.shape}")

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# Configuración del generador de variaciones geométricas
datagen = ImageDataGenerator(
    rotation_range=15,      # Rotaciones leves de hasta 15 grados
    width_shift_range=0.1,  # Desplazamiento horizontal del 10%
    height_shift_range=0.1, # Desplazamiento vertical del 10%
    zoom_range=0.1,         # Zoom aleatorio de hasta el 10%
    horizontal_flip=False   # Falso: invertir horizontalmente podría alterar la semántica de la seña
)

# Ajustamos el generador exclusivamente con los datos de entrenamiento
datagen.fit(X_train)
print("¡Pipeline de Data Augmentation configurado exitosamente!")